In [ ]:
# Import necessary libraries
from pathlib import Path
from PIL import Image, ImageDraw, ImageFilter
import random

# input folders
images_folder = Path("Training Data")
bg_folder = Path("Backgrounds")

# output folders
output_path = Path("dataset")
train_images = output_path / "images" / "train"
train_labels = output_path / "labels" / "train"
val_images = output_path / "images" / "val"
val_labels = output_path / "labels" / "val"

train_images.mkdir(parents=True, exist_ok=True)
train_labels.mkdir(parents=True, exist_ok=True)
val_images.mkdir(parents=True, exist_ok=True)
val_labels.mkdir(parents=True, exist_ok=True)

# get original pictures
items = {}
for folder in images_folder.iterdir(): # images_folder stores a folder for each class
    temp = []
    for i in folder.iterdir(): # each folder contains the actual images
        temp.append(Path(i))
    items[folder] = temp

bgs = []
for bg in bg_folder.iterdir():
    bgs.append(bg)
    
# class names = name of product, folder name = class name
class_folders = list(items.keys())
class_names = []
for folder in class_folders:
    class_names.append(folder.name)
class_names.sort()

# convert each class to an id for YOLO, save them in classes.txt
class_to_id = {}
for i in range(len(class_names)):
    class_to_id[class_names[i]] = i

classes_file = open(output_path / "classes.txt", "w")
classes_file.write("\n".join(class_names)) # writes each class name on its own line
classes_file.close()

# ensures that pasted in images do not overlap
def verify_location(existing_boxes, new_left, new_top, w, h):
    new_right = new_left + w
    new_bottom = new_top + h
    for box in existing_boxes:
        box_left, box_top, box_right, box_bottom = box # unpacks coordinates from box
        
        if new_right > box_left and new_left < box_right: # not completely left OR not completely right
            if new_bottom > box_top and new_top < box_bottom: # not completely on top OR not completely below
                return False
    return True

# scale each product down, rotate it, then remove added transparent pixels from the rotation
def transform_image(img):
    target = random.uniform(0.2, 0.8) * 640 # 20-80% of background
    scale = target / max(img.width, img.height) # scale = divide target by the LONGEST side of image | if (400,200), target = 300, scale = 300/400 = 0.75
    # new dimensions
    new_w = max(1, int(img.width * scale))
    new_h = max(1, int(img.height * scale))
    img = img.resize((new_w, new_h), Image.LANCZOS)

    # rotation
    if random.random() < 0.7:
        angle = random.uniform(-30, 30)   # 70% near-upright since 2k images isn't enough to cover 360 well, so prioritised upright which is more common
    else:
        angle = random.uniform(0, 360)
    img = img.rotate(angle, expand=True, resample=Image.BICUBIC) # bicubic = interpolation method, rotating the image causes the pixels to not align with a grid
    img = img.crop(img.getbbox()) # expand=True above causes the cavnas to grow to fit the rotated shape, crop + bbox trims off the transparent pixels
    return img

# place a product on the background, tries 100 times to place it in different locations without overlap
def place_item(bg, existing_boxes, labels, bg_w, bg_h):
    random_class = random.choice(class_folders) 
    random_path = random.choice(items[random_class])
    img = Image.open(random_path).convert("RGBA") # RGBA allows for transparent pixels, otherwise the transparent pixels become solid black or white
    img = transform_image(img) # apply transformations
    
    # ensure transformed image isn't larger than the background
    w, h = img.size
    if w >= bg_w or h >= bg_h:
        return False
        
    # try 100 times to place image in a random spot
    for attempt in range(100):
        # random placement location, without going off the background
        random_left = random.randint(0, bg_w - w)
        random_top = random.randint(0, bg_h - h)
        
        if verify_location(existing_boxes, random_left, random_top, w, h):
            bg.paste(img, (random_left, random_top), img) # paste onto background image
            
            # store product coordinates and dimensions
            box = (random_left, random_top, random_left + w, random_top + h) 
            existing_boxes.append(box)
            
            # save label and coordinate of product as a tuple 
            labels.append((random_class.name, box))
            return True
    return False

# generate labels for YOLO
def make_label_text(labels, bg_w, bg_h):
    lines = []
    for class_name, box in labels:
        x1, y1, x2, y2 = box

        x_center = ((x1 + x2) / 2) / bg_w
        y_center = ((y1 + y2) / 2) / bg_h
        width = (x2 - x1) / bg_w
        height = (y2 - y1) / bg_h

        class_id = class_to_id[class_name]
        line = f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}"
        lines.append(line)

    return "\n".join(lines) # one label per line

# generate training data
def generate(split, count):
    img_dir = output_path / "images" / split
    lbl_dir = output_path / "labels" / split

    for i in range(count):
        existing_boxes = []
        labels = []

        # get random background 
        random_bg = random.choice(bgs)
        bg_img = Image.open(random_bg).convert("RGBA")

        # get a random 640x640 piece from the background
        src_w, src_h = bg_img.size
        crop_x = random.randint(0, src_w - 640)
        crop_y = random.randint(0, src_h - 640)
        bg_img = bg_img.crop((crop_x, crop_y, crop_x + 640, crop_y + 640))

        # place items on background
        num_items = random.randint(1, 7)
        for j in range(num_items):
            place_item(bg_img, existing_boxes, labels, 640, 640)

        # add shadow
        if random.random() < 0.5:
            sh_size = random.uniform(0.4, 1.2) # scale, can be bigger that the image
            sh_x1 = random.randint(-320, 640) # allows it to start off the picture
            sh_y1 = random.randint(-320, 640)
            sh_x2 = sh_x1 + int(640 * sh_size)
            sh_y2 = sh_y1 + int(640 * sh_size)

            mask = Image.new("L", (640, 640), 0) # grayscale mask (L = 0-255)
            ImageDraw.Draw(mask).ellipse([sh_x1, sh_y1, sh_x2, sh_y2], fill=random.randint(60, 130)) # fill = darkness of shadow
            mask = mask.filter(ImageFilter.GaussianBlur(random.randint(20, 60))) # blurs edges randomly so it looks less like an ellipse

            black = Image.new("RGBA", (640, 640), (0, 0, 0, 255)) # generate a fully black image of the image size
            bg_img.paste(black, (0, 0), mask) # then paste the black through the mask 

        # save final image
        image_name = f"{split}_{i:04d}.jpg" # 04d pads to 4 digits, like 0001 instead of 1
        label_name = f"{split}_{i:04d}.txt"

        bg_img.convert("RGB").save(img_dir / image_name) # jpg doesn't support transparency, but jpg files are smaller in size

        # write label files for each image, each file contains the classes and bounding boxes of every item in that image
        label_text = make_label_text(labels, 640, 640) 
        label_file = open(lbl_dir / label_name, "w")
        label_file.write(label_text)
        label_file.close()

        # print progress
        if (i + 1) % 20 == 0:
            print(f"{split}: {i + 1}/{count}")


generate("train", 2000)
generate("val", 300)]
print("done")